In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from riesgo_credito.procesamiento import esquema, etl

TRIMESTRE = "2019Q1"
DATOS = RAIZ / "data"
RUTAS = {
    "originacion": DATOS / "limpios" / "originacion" / f"trimestre={TRIMESTRE}",
    "desempeno": DATOS / "limpios" / "desempeno" / f"trimestre={TRIMESTRE}",
    "p1": DATOS / "modelado" / "p1" / f"trimestre={TRIMESTRE}",
}

metricas = json.loads((DATOS / "metricas" / "etl" / f"{TRIMESTRE}.json").read_text())
orig = pd.read_parquet(RUTAS["originacion"])
p1 = pd.read_parquet(RUTAS["p1"])
perf = pd.read_parquet(RUTAS["desempeno"], columns=["loan_id", "period", "edad"])
pd.Series(metricas, name="valor").to_frame()

,valor
trimestre,2019Q1
creditos,279861
registros_desempeno,9821276
periodo_inicial,201901
fecha_corte,202603
excluidos_codigo_96,801
p1_creditos,279060
p1_etiquetables,279060
p1_positivos,680


## Cifras de referencia del EDA

In [2]:
referencia = {"creditos": 279861, "registros_desempeno": 9821276, "excluidos_codigo_96": 801,
              "p1_etiquetables": 279060, "p1_positivos": 680}
comparacion = pd.DataFrame({"eda": referencia, "etl": {k: metricas[k] for k in referencia}})
comparacion["coincide"] = comparacion.eda == comparacion.etl
assert comparacion.coincide.all()
comparacion

,eda,etl,coincide
creditos,279861,279861,True
registros_desempeno,9821276,9821276,True
excluidos_codigo_96,801,801,True
p1_etiquetables,279060,279060,True
p1_positivos,680,680,True


El ETL en Spark reproduce exactamente las cifras del EDA, así que la lectura, la limpieza, la exclusión del código 96 y la etiqueta de la pregunta 1 quedaron bien portadas.

## Calidad de las salidas

In [3]:
sin_defecto = set(orig.loc[~orig.defecto, "loan_id"])
chequeos = pd.Series({
    "duplicados en originación": orig.loan_id.duplicated().sum(),
    "duplicados en desempeño (crédito-mes)": perf.duplicated(["loan_id", "period"]).sum(),
    "duplicados en P1": p1.loan_id.duplicated().sum(),
    "créditos sin defecto que faltan en P1": len(sin_defecto - set(p1.loan_id)),
    "créditos con código 96 en P1": len(set(p1.loan_id) - sin_defecto),
    "columnas posteriores al cierre en desempeño": len(set(esquema.POSTERIORES_AL_CIERRE) & set(pd.read_parquet(RUTAS["desempeno"]).columns)),
    "columnas fuera de la solicitud en P1": len(set(p1.columns) - set(etl.VARIABLES_P1) - {"loan_id", "first_payment_date", "y_p1", "trimestre"}),
    "puntajes 9999 en P1": (p1.credit_score == 9999).sum(),
    "deuda-ingreso 999 en P1": (p1.dti == 999).sum(),
    "registros previos al primer pago (edad <= 0)": (perf.edad <= 0).sum(),
}, name="valor")
chequeos.to_frame()

,valor
duplicados en originación,0
duplicados en desempeño (crédito-mes),0
duplicados en P1,0
créditos sin defecto que faltan en P1,0
créditos con código 96 en P1,0
columnas posteriores al cierre en desempeño,0
columnas fuera de la solicitud en P1,0
puntajes 9999 en P1,0
deuda-ingreso 999 en P1,0
registros previos al primer pago (edad <= 0),255425


In [4]:
nulos = (100 * p1.isna().mean()).round(3)
nulos[nulos > 0].rename("pct_nulos").to_frame()

,pct_nulos
credit_score,0.054
dti,0.400
ocltv,0.003


Las llaves son únicas y la tabla de modelado no tiene columnas posteriores al cierre, códigos de no disponible ni créditos con código 96. Los faltantes de P1 son pocos y se dejan como nulos para que cada modelo decida cómo tratarlos. Los registros con edad cero o negativa son los meses previos al primer pago y se conservan en la capa limpia, como en el EDA.

## Tablas generadas

In [5]:
tamanos = {n: sum(f.stat().st_size for f in r.glob("*.parquet")) / 1e6 for n, r in RUTAS.items()}
columnas = {"originacion": orig.shape[1], "desempeno": len(pd.read_parquet(RUTAS["desempeno"]).columns), "p1": p1.shape[1]}
filas = {"originacion": len(orig), "desempeno": len(perf), "p1": len(p1)}
pd.DataFrame({"filas": filas, "columnas": columnas, "mb": tamanos}).round(1)

,filas,columnas,mb
originacion,279861,27,5.0
desempeno,9821276,25,123.9
p1,279060,19,4.1
